# NLP: Question Answering from PDF
### Transformer-Based Extractive QA with Relevance Checking & Answerability Gating
**Course**: Machine Learning / NLP — Final Project

This notebook implements an extractive QA pipeline using Transformer models (`distilbert-base-cased-distilled-squad`).
- **PDF Extraction**: Selectable text extraction with page numbering using `PyMuPDF` (`fitz`).
- **Sliding Context Window**: Overlapping word chunks (`words_per_chunk=180`, `overlap=40`).
- **Relevance Filter**: Content-word BOW overlap prevents model hallucinations.
- **Answerability Gate**: Returns `"Answer not found in the uploaded PDF."` when evidence is below threshold.
- **Granular Attribution**: Returns exact source filename and page number (`Source: Account_Security.pdf | Page 1`).
- **Backend API Sync**: Connects with the PHP Laravel / FastAPI backend.


In [ ]:
# Environment Setup
!pip -q install pymupdf transformers sentencepiece scikit-learn torch gradio
print("Libraries installed successfully.")


In [ ]:
import os
import re
from pathlib import Path
import fitz
import torch
from transformers import AutoTokenizer, AutoModelForQuestionAnswering

print("Modules imported successfully. PyTorch version:", torch.__version__)


## 1. Upload PDF file(s)


In [ ]:
# Upload PDF file(s) via Google Colab, or use default sample file
uploaded_pdf_files = []

try:
    from google.colab import files
    print("Select one or more PDF files from your computer:")
    uploaded = files.upload()
    uploaded_pdf_files = [
        filename for filename in uploaded.keys()
        if filename.lower().endswith(".pdf")
    ]
except Exception:
    print("Not running in interactive Colab upload mode, checking local sample files...")

# Fallback: create Account_Security.pdf if not uploaded
if not uploaded_pdf_files:
    sample_pdf_path = "Account_Security.pdf"
    if not os.path.exists(sample_pdf_path):
        doc = fitz.open()
        page = doc.new_page()
        page.insert_text((50, 72), """ACCOUNT SECURITY AND PASSWORD POLICY

1. Password Reset Policy:
According to the "Account Security" document, you can reset your password by clicking the "Forgot Password" link on the login page and following the instructions sent to your registered email.

2. Two-Factor Authentication (2FA):
Two-Factor Authentication is strongly recommended for all accounts. When enabled, users must provide a 6-digit one-time verification passcode in addition to their password during login.

3. Account Lockout Policy:
After 5 consecutive failed login attempts, your account will be temporarily locked for 15 minutes to protect against brute-force attacks.
""", fontsize=12)
        doc.save(sample_pdf_path)
        doc.close()
    uploaded_pdf_files = [sample_pdf_path]

print("\nUploaded PDF files:")
for filename in uploaded_pdf_files:
    print("✓", filename)
print(f"\nTotal PDFs uploaded: {len(uploaded_pdf_files)}")


## 2. Extract PDF text with page numbers


In [ ]:
def normalize_pdf_text(text):
    # Normalize repeated OCR/PDF extraction noise without using any domain-specific data.
    text = re.sub(r'[_$%&]{3,}', ' ', text)
    text = re.sub(r'[()]{5,}', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

def extract_pdf_pages(pdf_path):
    # Extract selectable text page-by-page from a PDF.
    doc = fitz.open(pdf_path)
    pages = []
    for page_number, page in enumerate(doc, start=1):
        text = normalize_pdf_text(page.get_text('text'))
        if text:
            pages.append({
                'file': Path(pdf_path).name,
                'page': page_number,
                'text': text
            })
    doc.close()
    return pages

pdf_pages = []
for pdf_file in uploaded_pdf_files:
    pdf_pages.extend(extract_pdf_pages(pdf_file))

if not pdf_pages:
    raise ValueError(
        'The uploaded PDFs did not contain selectable text. '
        'Use PDFs with text or add OCR as a separate preprocessing step.'
    )

print(f'Extracted text from {len(pdf_pages)} page(s).')
for item in pdf_pages:
    print(f"✓ {item['file']} — Page {item['page']}")


## 3. Create overlapping context chunks
Long PDF pages are divided into overlapping word windows so that the extractive model does not truncate context.


In [ ]:
def make_chunks(records, words_per_chunk=180, overlap=40):
    chunks = []
    for record in records:
        words = record["text"].split()
        if not words:
            continue
        start = 0
        while start < len(words):
            end = min(start + words_per_chunk, len(words))
            chunks.append({
                "file": record["file"],
                "page": record["page"],
                "text": " ".join(words[start:end])
            })
            if end == len(words):
                break
            start = end - overlap
    return chunks

chunks = make_chunks(pdf_pages)
print(f"Created {len(chunks)} context chunks.")


## 4. Load the Transformer extractive QA model


In [ ]:
MODEL_NAME = "distilbert-base-cased-distilled-squad"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
qa_model = AutoModelForQuestionAnswering.from_pretrained(MODEL_NAME)
qa_model.to(device)
qa_model.eval()

print("Model loaded:", MODEL_NAME)
print("Device:", device)


In [ ]:
def content_words(text):
    """Return simple BOW-style content words for relevance checking."""
    stop_words = {
        "a", "an", "the", "is", "are", "was", "were", "be", "been",
        "being", "am", "do", "does", "did", "in", "on", "at", "to",
        "of", "for", "from", "with", "by", "and", "or", "but", "as",
        "it", "this", "that", "these", "those", "what", "which", "who",
        "whom", "whose", "where", "when", "why", "how", "can", "could",
        "would", "should", "will", "shall", "may", "might", "i", "you",
        "your", "we", "they", "he", "she", "them", "his", "her", "their"
    }
    words = re.findall(r"[A-Za-z0-9]+", text.lower())
    return [word for word in words if word not in stop_words and len(word) > 1]

def relevance_score(question, context):
    """
    BOW/content-word relevance check.
    This prevents the extractive model from inventing an answer from an
    unrelated context when the question topic is absent from the PDF.
    """
    question_words = set(content_words(question))
    context_words = set(content_words(context))
    if not question_words:
        return 0.0
    overlap = question_words.intersection(context_words)
    return len(overlap) / len(question_words)

def extract_answer(question, context):
    """Find the highest-scoring answer span using start/end QA logits."""
    inputs = tokenizer(
        question,
        context,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )
    inputs = {key: value.to(device) for key, value in inputs.items()}

    with torch.no_grad():
        outputs = qa_model(**inputs)

    start_logits = outputs.start_logits[0]
    end_logits = outputs.end_logits[0]

    start_index = int(torch.argmax(start_logits).item())
    end_index = int(torch.argmax(end_logits).item())

    if end_index < start_index:
        end_index = start_index

    max_answer_tokens = 40
    if end_index - start_index + 1 > max_answer_tokens:
        end_index = start_index + max_answer_tokens - 1

    answer_tokens = inputs["input_ids"][0][start_index:end_index + 1]
    answer = tokenizer.decode(
        answer_tokens,
        skip_special_tokens=True
    ).strip()

    start_prob = torch.softmax(start_logits, dim=0)[start_index]
    end_prob = torch.softmax(end_logits, dim=0)[end_index]
    qa_score = float((start_prob * end_prob).item())

    return answer, qa_score


In [ ]:
def answer_single_question(question, top_k=3):
    """Answer one question only when the uploaded PDF contains relevant evidence."""
    if not question or not question.strip():
        raise ValueError("Please enter a question.")

    candidates = []
    for chunk in chunks:
        relevance = relevance_score(
            question,
            chunk["text"]
        )

        # Do not send clearly unrelated chunks to the QA model.
        if relevance < 0.20:
            continue

        answer, qa_score = extract_answer(
            question.strip(),
            chunk["text"]
        )

        if answer:
            candidates.append({
                "answer": answer,
                "score": qa_score,
                "relevance": relevance,
                "file": chunk["file"],
                "page": chunk["page"],
                "context": chunk["text"]
            })

    if not candidates:
        return [{
            "answer": "Answer not found in the uploaded PDF.",
            "score": 0.0,
            "relevance": 0.0,
            "file": None,
            "page": None,
            "context": None
        }]

    # Prefer evidence relevance first, then QA confidence.
    candidates.sort(
        key=lambda item: (item["relevance"], item["score"]),
        reverse=True
    )

    best = candidates[0]

    # Final answerability gate.
    # If the question has weak evidence in the uploaded PDF, do not guess.
    if best["relevance"] < 0.20 or best["score"] < 0.05:
        return [{
            "answer": "Answer not found in the uploaded PDF.",
            "score": best["score"],
            "relevance": best["relevance"],
            "file": best["file"],
            "page": best["page"],
            "context": best["context"]
        }]

    return candidates[:top_k]

def split_compound_question(question):
    """Split a prompt containing multiple independent questions."""
    parts = [
        part.strip()
        for part in re.split(r"\?+", question)
        if part.strip()
    ]
    return [
        part + "?" if not part.endswith("?") else part
        for part in parts
    ]

def answer_question(question, top_k=3):
    """Answer a single or compound question from the uploaded PDF(s)."""
    subquestions = split_compound_question(question)
    all_results = []
    for subquestion in subquestions:
        results = answer_single_question(
            subquestion,
            top_k=top_k
        )
        best = results[0]
        best["question"] = subquestion
        all_results.append(best)
    return all_results

def display_answer(question):
    results = answer_question(question)
    for result in results:
        print("\nQuestion:", result["question"])
        print("Answer  :", result["answer"])
        if result["file"] is not None:
            print("Score   :", round(result["score"], 4))
            print("Evidence:", round(result["relevance"], 4))
            print("Source  :", result["file"], "| Page", result["page"])
        else:
            print("Evidence: No relevant text found in the uploaded PDF.")
    return results


## 5. Interactive PDF Question Answering
Enter any question whose answer is contained in the uploaded PDF(s).
The system first checks whether the question's content words are represented in the uploaded PDF context. If there is insufficient evidence, it returns:
`Answer not found in the uploaded PDF.`


In [ ]:
# Run sample test queries
print("--- TEST 1: In-scope Query ---")
display_answer("How do I reset my password?")

print("\n--- TEST 2: In-scope Query (Lockout Policy) ---")
display_answer("What is the account lockout policy?")

print("\n--- TEST 3: Out-of-scope Query ---")
display_answer("What is the recipe for chocolate cake?")


## 6. Interactive Chat UI (Matching Mobile / Phone Widget)


In [ ]:
import gradio as gr

def chat_interface_fn(user_query, history):
    results = answer_question(user_query)
    best = results[0]
    
    if best["file"] is not None:
        response = (
            f"{best['answer']}\n\n"
            f"---\n"
            f"**Source**: `{best['file']}` | **Page**: {best['page']} "
            f"(Confidence: {best['score']*100:.1f}%, Evidence: {best['relevance']*100:.1f}%)"
        )
    else:
        response = (
            f"⚠️ **{best['answer']}**\n\n"
            "I can only answer questions grounded in the uploaded PDF documents."
        )
    return response

demo = gr.ChatInterface(
    fn=chat_interface_fn,
    title="📱 KnowledgeBot - PDF Question Answering",
    description="Grounded Question Answering from uploaded PDF files. Matches the mobile phone UI.",
    examples=[
        ["How do I reset my password?"],
        ["What is the account lockout policy?"],
        ["What is Two-Factor Authentication?"],
        ["Who won the football world cup?"]
    ]
)

demo.launch(share=True, debug=False)


## 7. Connect to Backend & Admin API (PHP / Laravel / FastAPI)
You can sync this notebook directly with your local or deployed backend!


In [ ]:
import requests

BACKEND_API_URL = "http://127.0.0.1:8000/api"  # or http://127.0.0.1:8080/api for PHP Laravel

def sync_with_backend(query):
    """Send query directly to the running backend service"""
    try:
        res = requests.post(
            f"{BACKEND_API_URL}/chat/message",
            json={"query": query, "top_k": 3},
            timeout=8
        )
        return res.json()
    except Exception as e:
        return {"error": f"Backend not reachable: {e}"}

print("Backend sync function ready. Call sync_with_backend('your question') when backend is running.")
